In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from limb_fitting import *
from imageutils import *
from lineutils import *
from calc_flatfield import *
from scipy.ndimage import gaussian_filter
from fittingutils import *
from cavity_correction import correct_cavity

In [2]:
folder = 'temp1/'
files = sorted(glob.glob(folder + '*.fits'))
files

['temp1/solo_L1_phi-fdt-ilam_20250310T080009_V202503131733C_0563100100.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T080608_V202503131733C_0563100125.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T081208_V202503131835C_0563100150.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T081808_V202503131935C_0563100175.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T082408_V202503131935C_0563100200.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T083008_V202503132033C_0563100225.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T083608_V202503141634C_0563100250.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T084208_V202503141734C_0563100275.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T084808_V202503141833C_0563100300.fits']

In [3]:
flatfield = np.load('transmittance.npz')['flatfield']
flatfield = np.nan_to_num(flatfield, nan=1)

plt.figure(figsize=(10,10))
plt.imshow(flatfield, vmin=0.9, vmax=1.1)
plt.tight_layout()

In [4]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

with fits.open(dark_file) as hdul:
    dark = hdul[0].data
    dark_header = hdul[0].header

cavity_file = '/home/ulyanov/data/solo/phi/cavity/phi-fdt-cavity_average_V202511261217C.fits'

with fits.open(cavity_file) as hdul:
    cavity = hdul[0].data

In [5]:
dark_header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / number of bits per data pixel                  
NAXIS   =                    2 / number of data axes                            
NAXIS1  =                 2048 / length of data axis 1                          
NAXIS2  =                 2048 / length of data axis 2                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
LONGSTRN= 'OGIP 1.0'           / The HEASARC Long String Convention may be used.
COMMENT   This FITS file may contain long string keyword values that are        
COMMENT   continued over multiple keywords.  The HEASARC convention uses the &  
COMMENT   character at the end of each substring which is then continued        
COMMENT   on the next keywor

In [6]:
images = []

for file in files[:]:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header
        data = data / flatfield
        data = realign(data, header=header)
        stokes = demodulate(data)

    images.append(stokes)

images = np.array(images)

In [10]:
header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / number of bits per data pixel                  
NAXIS   =                    3 / number of data axes                            
NAXIS1  =                 2048 / length of data axis 1                          
NAXIS2  =                 2048 / length of data axis 2                          
NAXIS3  =                    4 / length of data axis 3                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
LONGSTRN= 'OGIP 1.0'           / The HEASARC Long String Convention may be used.
COMMENT   This FITS file may contain long string keyword values that are        
COMMENT   continued over multiple keywords.  The HEASARC convention uses the &  
COMMENT   character at the e

In [11]:
480 * 0.013

6.239999999999999

In [118]:
I = images[:,0]
Q = images[:,1]
xr, yr = calc_reflection_center(I, Q)

In [119]:
I = images[:,0]
Q = images[:,1]

F, G = calc_polarization(I, Q, xr, yr)

/tmp/ipykernel_59819/1946135061.py:36: RuntimeWarning: divide by zero encountered in divide
  F = a / b
/tmp/ipykernel_59819/1946135061.py:37: RuntimeWarning: invalid value encountered in multiply
  W = I / np.abs(Q - F * I - G * I_).clip(sigma)


In [120]:
plt.figure(figsize=(10,10))
plt.imshow(F, vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [121]:
plt.figure(figsize=(10,10))
plt.imshow(G, vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [122]:
folder = '/home/ulyanov/data/solo/phi/test/'
#folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-01-19/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240101T040003_V202401090117C_0441010503.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240106T210007_V202401100517C_0441060508.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240107T000009_V202401110118C_0441070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T000009_V202402130123C_0442070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T060009_V202402130123C_0442070502.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240318T190009_V202405151841C_0443180504.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240328T060009_V202405152307C_0443281521.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T044009_V202405152319C_0443301531.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T233009_V202405152329C_0443300501.fits.gz',
 '/home/ulyanov/data/solo/ph

In [123]:
def reflection_point_predict(header, **kwargs):

    px = [1.63114715e-06, 6.72511045e-03, 9.60448053e+02]
    py = [ 4.61830880e-06, -6.85005911e-03,  9.77508840e+02]


    r_sun = header['RSUN_ARC'] #/ header['CDELT1']
    dx, dy = header['PXBEG2'] - 1, header['PXBEG1'] - 1

    xr = np.polyval(px, r_sun) - dx
    yr = np.polyval(py, r_sun) - dy
    return xr, yr

In [134]:
with fits.open(files[-5]) as hdul:
    data = hdul[0].data
    header = hdul[0].header

data -= 0.4 * crop(dark, header)
data = data / crop(flatfield, header)

cpos = header['CONTPOS'] - 1
print(cpos)


wv = []
for i in range(6):
    wv.append(header[f'WAVELN{i+1:02d}'])
wv = np.array(wv)

data = correct_cavity(data, wv, crop(cavity, header), c_pos=cpos)

nx, ny = data.shape[-2:]
xr, yr = reflection_point_predict(header)

5


In [135]:
p = calc_ghost_scaling(data, cpos=cpos)

In [136]:
i = 0

data_ = data.reshape(6,4,nx,ny)[i].copy()
data_ = demodulate(data_)

plt.figure(figsize=(10,10))
plt.imshow(data_[1] - crop(F, header) * data_[0] - crop(G, header) * reflect(gaussian_filter(data_[0], 8) * p[i], xr, yr), vmin=-30, vmax=30)
plt.tight_layout()

In [205]:
plt.figure(figsize=(10,10))
plt.imshow(p[4], vmin=0.9, vmax=1.3)
plt.tight_layout()